# 00 — Raw Recording Preprocessing

**Purpose:** Convert long raw recordings (2–10 min per speaker) into 3-second training clips.

## Expected input layout

```
data/raw/
  baraka/
    recording_01.wav   ← any length, any sample rate
    recording_02.mp3
  alice/
    reading.wav
  ...
```

Supported formats: `.wav`, `.mp3`, `.flac`, `.ogg`, `.m4a` — anything ffmpeg handles.

## Output

```
data/recordings/<speaker>/<NNNN>.wav   ← 3 s, 16 kHz mono
```

Run **notebooks 01 and 02** after this to train the classifiers.

## 1. Configuration

In [ ]:
import sys
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf

ROOT      = Path().resolve().parent
RAW_DIR   = ROOT / 'data' / 'raw'
OUT_DIR   = ROOT / 'data' / 'recordings'

SR          = 16_000   # target sample rate
CLIP_SECS   = 3.0      # clip length in seconds
CLIP_SAMPLES = int(SR * CLIP_SECS)
HOP_SECS    = 1.5      # stride between clips (50 % overlap)
HOP_SAMPLES = int(SR * HOP_SECS)

# Drop clips whose RMS energy is below this fraction of the speaker's mean —
# removes long silence segments from between sentences.
SILENCE_THRESHOLD = 0.10  # 10 % of mean RMS → discard

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Raw dir:  {RAW_DIR}')
print(f'Out dir:  {OUT_DIR}')
print(f'Clip length: {CLIP_SECS} s  |  Hop: {HOP_SECS} s')

## 2. Load and inspect raw recordings

Each speaker sub-directory under `data/raw/` is one class. We list what's available before processing.

In [ ]:
AUDIO_EXTS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a', '.aac'}

if not RAW_DIR.exists():
    raise RuntimeError(
        f'{RAW_DIR} does not exist.\n'
        'Create it and place raw recordings inside per-speaker subdirectories.\n'
        '  data/raw/<speaker_name>/<recording>.<ext>'
    )

speakers = sorted([d for d in RAW_DIR.iterdir() if d.is_dir()])
if not speakers:
    raise RuntimeError(f'No speaker directories found in {RAW_DIR}')

print(f'Found {len(speakers)} speakers:')
total_files = 0
for spk in speakers:
    files = [f for f in spk.iterdir() if f.suffix.lower() in AUDIO_EXTS]
    total_files += len(files)
    print(f'  {spk.name}: {len(files)} file(s) → ', end='')
    for f in files:
        try:
            y, sr = librosa.load(str(f), sr=None, mono=True, duration=5)
            dur = librosa.get_duration(y=y, sr=sr)
            print(f'{f.name} ({dur:.1f}s)', end='  ')
        except Exception as e:
            print(f'{f.name} (ERROR: {e})', end='  ')
    print()
print(f'\nTotal raw files: {total_files}')

## 3. Preprocessing pipeline

For each raw file:
1. **Resample** to 16 kHz mono
2. **Normalise** amplitude (peak normalise to ±0.9)
3. **Chunk** with a sliding window (3 s clip, 1.5 s hop)
4. **VAD filter** — discard clips whose RMS is below the silence threshold
5. **Save** as 16-bit PCM WAV

In [ ]:
def rms(y: np.ndarray) -> float:
    return float(np.sqrt(np.mean(y ** 2)))


def process_speaker(spk_dir: Path, out_dir: Path) -> dict:
    files = [f for f in spk_dir.iterdir() if f.suffix.lower() in AUDIO_EXTS]
    if not files:
        return {'clips': 0, 'dropped': 0, 'duration_s': 0}

    out_dir.mkdir(parents=True, exist_ok=True)
    clip_idx   = len(list(out_dir.glob('*.wav')))  # resume from where we left off
    total_kept = 0
    total_drop = 0
    total_dur  = 0.0

    for fpath in sorted(files):
        try:
            y, _ = librosa.load(str(fpath), sr=SR, mono=True)
        except Exception as e:
            print(f'    SKIP {fpath.name}: {e}')
            continue

        total_dur += len(y) / SR

        # Peak normalise
        peak = np.max(np.abs(y))
        if peak > 0:
            y = y / peak * 0.9

        # Compute per-clip RMS for VAD threshold
        all_clips = []
        start = 0
        while start + CLIP_SAMPLES <= len(y):
            all_clips.append(y[start : start + CLIP_SAMPLES])
            start += HOP_SAMPLES

        if not all_clips:
            continue

        rms_vals  = np.array([rms(c) for c in all_clips])
        rms_mean  = rms_vals.mean()
        threshold = rms_mean * SILENCE_THRESHOLD

        for clip, rv in zip(all_clips, rms_vals):
            if rv < threshold:
                total_drop += 1
                continue
            dest = out_dir / f'{clip_idx:04d}.wav'
            sf.write(dest, clip.astype(np.float32), SR)
            clip_idx  += 1
            total_kept += 1

    return {'clips': total_kept, 'dropped': total_drop, 'duration_s': total_dur}


# ── Run ───────────────────────────────────────────────────────────
summary = {}
for spk in speakers:
    out = OUT_DIR / spk.name
    print(f'{spk.name} ...', end=' ', flush=True)
    stats = process_speaker(spk, out)
    summary[spk.name] = stats
    print(f'{stats["clips"]} clips kept  ({stats["dropped"]} silent dropped)  '
          f'from {stats["duration_s"]:.0f} s of audio')

print(f'\nTotal clips: {sum(s["clips"] for s in summary.values())}')

## 4. Visual quality check

Plot a random waveform and log-mel spectrogram from each speaker to confirm the clips look reasonable.

In [ ]:
import random

n_speakers = len(speakers)
fig, axes = plt.subplots(n_speakers, 2,
                          figsize=(12, 2.5 * n_speakers),
                          facecolor='#111')
if n_speakers == 1:
    axes = [axes]

for i, spk in enumerate(speakers):
    wavs = sorted((OUT_DIR / spk.name).glob('*.wav'))
    if not wavs:
        continue
    sample = random.choice(wavs)
    y, _   = librosa.load(str(sample), sr=SR)

    ax_w, ax_s = axes[i]

    # Waveform
    librosa.display.waveshow(y, sr=SR, ax=ax_w, color='#fff', alpha=0.7)
    ax_w.set_title(f'{spk.name} — waveform', color='#aaa', fontsize=9)
    ax_w.set_facecolor('#111'); ax_w.tick_params(colors='#555')
    ax_w.set_xlabel(''); ax_w.set_ylabel('')

    # Spectrogram
    mel = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=64, n_fft=512, hop_length=160)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    librosa.display.specshow(log_mel, sr=SR, x_axis='time', y_axis='mel',
                              ax=ax_s, cmap='magma')
    ax_s.set_title(f'{spk.name} — log-mel', color='#aaa', fontsize=9)
    ax_s.set_facecolor('#111'); ax_s.tick_params(colors='#555')
    ax_s.set_xlabel(''); ax_s.set_ylabel('')

plt.tight_layout()
plt.savefig(ROOT / 'models' / 'preprocess_samples.png', dpi=100, facecolor='#111')
plt.show()
print('Saved to models/preprocess_samples.png')

## 5. Clip count summary

Check the balance across speakers. Very imbalanced classes hurt SVM more than CNN (CNN can use augmentation).

In [ ]:
print('Speaker           Clips   Duration (raw)')
print('-' * 45)
for spk in speakers:
    wavs = list((OUT_DIR / spk.name).glob('*.wav'))
    stats = summary.get(spk.name, {})
    raw_s = stats.get('duration_s', 0)
    print(f'{spk.name:<18} {len(wavs):>5}   {raw_s:.0f} s ({raw_s/60:.1f} min)')

print()
print('If any speaker has < 30 clips, record more or lower HOP_SECS.')
print('Aim for ≥ 50 clips per speaker.')
print()
print('Next: run notebooks/01_classical_pipeline.ipynb and 02_neural_pipeline.ipynb')